# EDA — ClawTrojan

**Dataset:** [`zstanjj/ClawTrojan`](https://huggingface.co/datasets/zstanjj/ClawTrojan)

Dataset de evaluación de seguridad de agentes: cada muestra es una **trayectoria** de varios pasos donde un ataque (prompt injection, exfiltración, escalamiento de privilegios…) entra por herramientas, archivos o memoria.

**Formato:** No es una tabla plana: `samples/` (metadatos por trayectoria, JSON), `steps/` (pasos por trayectoria, JSONL), `envs/` (entornos ejecutables por paso), `user_profiles/`, `skills_bundles/`. **Gran parte del texto está en chino.**

**Preguntas guía para el EDA**
- ¿Cómo se distribuyen escenarios, tipos de ataque, nivel de riesgo y resultado?
- ¿Cuántos pasos tiene cada trayectoria y en qué etapa entra la inyección?
- ¿Por qué canal entra la inyección (herramienta, archivo, memoria…)?
- ¿Cuántas muestras positivas vs negativas/borde hay?

In [ ]:
import sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Busca la raíz del repo (donde está src/profiler.py) y la agrega al path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "profiler.py").exists())
sys.path.insert(0, str(ROOT))
from src.profiler import *

sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

ctx = eda_context("08")   # rutas: datos en data/raw/..., resultados en esta carpeta
print(ctx)

## 1. Inventario (por carpeta; `envs/` tiene miles de archivos)

In [ ]:
for sub in sorted(p for p in ctx.data_dir.iterdir() if p.is_dir() and not p.name.startswith(".")):
    n = sum(1 for f in sub.rglob("*") if f.is_file())
    mb = sum(f.stat().st_size for f in sub.rglob("*") if f.is_file()) / 1e6
    print(f"{sub.name:15s} {n:>8,} archivos  {mb:8.1f} MB")

## 2. Muestras (trayectorias)

In [ ]:
recs = []
for p in sorted((ctx.data_dir / "samples").glob("*/*.json")):
    r = json.loads(p.read_text(encoding="utf-8"))
    r["carpeta"] = p.parent.name
    recs.append(r)
samples = pd.DataFrame(recs)
keep = ["sample_id", "carpeta", "scenario", "attack_type", "risk_tier", "outcome_category",
        "user_profile_id", "workspace_template", "skills_bundle_id", "annotation_status", "attack_goal", "goal"]
samples = samples[[c for c in keep if c in samples.columns]]
print(samples.shape)
display(samples.head())
display(profile_df(samples))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, c in zip(axes.ravel(), ["scenario", "attack_type", "risk_tier", "outcome_category"]):
    samples[c].value_counts().plot.barh(ax=ax); ax.set_title(c); ax.invert_yaxis()
plt.tight_layout(); save_fig(fig, ctx, "distribucion_muestras"); plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pd.crosstab(samples["attack_type"], samples["outcome_category"]), annot=True, fmt="d", cmap="Reds", ax=ax)
ax.set_title("Tipo de ataque vs. resultado"); save_fig(fig, ctx, "ataque_vs_resultado"); plt.show()

## 3. Pasos de cada trayectoria

In [ ]:
step_rows = []
for p in sorted((ctx.data_dir / "steps").glob("*/*.jsonl")):
    for r in iter_jsonl(p):
        r["carpeta"] = p.parent.name
        step_rows.append(r)
steps = pd.DataFrame(step_rows)
# Algunos campos vienen como texto ("None", "False"): se normalizan
steps["step_idx"] = pd.to_numeric(steps["step_idx"], errors="coerce")
steps["is_last_chance"] = steps["is_last_chance"].astype(str).str.lower().eq("true")
steps["injected_inst"] = steps["injected_inst"].replace({"None": None, "": None})
print(steps.shape)
display(steps.head(3))
display(profile_df(steps))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
steps.groupby("sample_id").size().value_counts().sort_index().plot.bar(ax=axes[0]); axes[0].set_title("Pasos por trayectoria")
steps["stage_tag"].value_counts().plot.bar(ax=axes[1]); axes[1].set_title("Etapa (stage_tag)")
steps["injection_src"].value_counts().plot.bar(ax=axes[2]); axes[2].set_title("Origen de la inyección")
plt.tight_layout(); save_fig(fig, ctx, "pasos"); plt.show()

inj = steps[steps["injected_inst"].notna()]
print(f"Pasos con inyección: {len(inj):,} de {len(steps):,}")
display(pd.crosstab(inj["stage_tag"], inj["injection_src"]))

## 4. Perfiles de usuario y entornos

In [ ]:
profiles = pd.DataFrame([json.loads(p.read_text(encoding="utf-8")) for p in sorted((ctx.data_dir / "user_profiles").glob("*.json"))])
display(profiles.drop(columns=[c for c in ["created_at", "updated_at"] if c in profiles]).head(10))
display(profiles[["role", "risk_tolerance", "security_awareness"]].value_counts().to_frame("n"))

env_steps = list((ctx.data_dir / "envs").glob("*/step_*"))
print(f"Entornos: {len({p.parent.name for p in env_steps}):,} trayectorias, {len(env_steps):,} pasos ejecutables")

### (Opcional) Usar el `loader.py` oficial
Convierte `envs/` en registros por paso con la salida de herramienta y la posición exacta de la inyección.

In [ ]:
RUN_LOADER = True
if RUN_LOADER:
    try:
        sys.path.insert(0, str(ctx.data_dir))
        from loader import load_all_trojan_envs
        env_records = pd.DataFrame([s.to_dict() for s in load_all_trojan_envs(str(ctx.data_dir / "envs"))])
        print(env_records.shape)
        display(env_records.drop(columns=["metadata", "contamination"], errors="ignore").head())
        display(env_records["is_malicious"].value_counts())
    except Exception as e:
        print("No se pudo usar loader.py:", e)

## 5. Ejemplos

In [ ]:
show_examples(steps, n=2, cols=["sample_id", "stage_tag", "user_input", "injected_inst", "injection_src"], max_chars=400)

## Guardar resultados
Se guardan en `tables/` de esta carpeta. `resumen.csv` lo consolida `eda/00_general`.

In [ ]:
save_table(profile_df(samples), ctx, "perfil_columnas")
display(save_summary(ctx, "zstanjj/ClawTrojan", samples, total_rows=None, dup_cols=["sample_id"],
                     notas="tabla de trayectorias (samples/)"))

## Próximos pasos / notas
- Anota tus hallazgos en el `README.md` de esta carpeta.
-